# Statistical evaluation of the representation study

This notebook evaluates **TTU** and **DOU** across the three between-subject groups while respecting that tasks are repeated within participants. It creates omnibus tests, pairwise comparisons, effect sizes, multiplicity-adjusted p-values, and compact letter displays.

## Why the analysis is structured this way

1. **A Kruskal-Wallis test has no within-subject argument.** Applying it to all 357 rows would treat repeated measurements as independent and therefore is not valid.
2. **Per-task comparisons are valid between-subject Kruskal-Wallis tests**, because each participant contributes only one observation to a task.
3. **Overall and structural-aspect comparisons first aggregate within participant** using the median. This yields one independent value per person and group.
4. **Omnibus effect size:** bias-corrected Kruskal-Wallis epsilon-squared.
5. **Pairwise effect size:** signed rank-biserial correlation. Positive means the first group tends to have higher values.
6. **Multiplicity:** Holm correction is applied across omnibus tests within each coherent family and within each stratum for the three pairwise comparisons.
7. **Compact letters:** groups sharing a letter are not significantly different after pairwise Holm correction. This is not proof of equivalence.
8. **DOU caution:** the severe ceiling effect reduces information and power.

In [1]:
#!/usr/bin/env python3
"""Robust statistical evaluation of the spreadsheet-comprehension study.

Run:
    python study_statistics.py --input study_data.csv --output results

Dependencies: pandas, numpy, scipy, statsmodels, matplotlib, seaborn
The script preserves independence by aggregating repeated task observations to
one value per participant before an overall or aspect-level Kruskal-Wallis test.
Per-task tests use one observation per participant directly.
"""
from __future__ import annotations

import argparse
import itertools
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

GROUPS = [
    "Dynamic sketch construction",
    "Formula construction",
    "Static sketch construction",
]
ASPECTS = ["COMP", "COND", "LOOK", "CELL", "RANGE", "LIT", "FUNC",
           "VAL", "BATCH", "LIST", "BACK", "FORWARD", "ON", "OFF", "CROSS"]
ALPHA = 0.05


def load_data(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    required = {"participant", "id", "group", "TTU", "DOU", "structural_aspects"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    df = df.copy()
    df["task"] = df["id"].astype(str).str.replace("task_", "", regex=False).astype(int)
    df["TTU"] = pd.to_numeric(df["TTU"], errors="coerce")
    df["DOU"] = pd.to_numeric(df["DOU"], errors="coerce")
    df = df[df["group"].isin(GROUPS)]
    if (df["TTU"].dropna() <= 0).any():
        raise ValueError("TTU must be greater than zero.")
    # A participant must belong to exactly one between-subject group.
    memberships = df.groupby("participant")["group"].nunique()
    if (memberships > 1).any():
        raise ValueError("At least one participant occurs in multiple groups.")
    if df.duplicated(["participant", "task"]).any():
        raise ValueError("Duplicate participant-task rows found.")
    return df.sort_values(["task", "group", "participant"]).reset_index(drop=True)


def holm(pvalues: pd.Series) -> np.ndarray:
    out = np.full(len(pvalues), np.nan)
    ok = pvalues.notna().to_numpy()
    if ok.any():
        out[ok] = multipletests(pvalues[ok], method="holm")[1]
    return out


def epsilon_squared_kw(H: float, n: int, k: int) -> float:
    """Bias-corrected Kruskal-Wallis epsilon squared, truncated to [0, 1]."""
    if n <= k:
        return np.nan
    return float(np.clip((H - k + 1) / (n - k), 0.0, 1.0))


def rank_biserial(x: np.ndarray, y: np.ndarray) -> float:
    """Signed rank-biserial correlation. Positive means x tends to exceed y."""
    u = stats.mannwhitneyu(x, y, alternative="two-sided", method="auto").statistic
    return float(2.0 * u / (len(x) * len(y)) - 1.0)


def compact_letters(groups: list[str], pairwise: pd.DataFrame, alpha: float = ALPHA) -> dict[str, str]:
    """Compact letter display based on Holm-adjusted pairwise p-values.

    Groups sharing a letter are not significantly different. Groups with no
    letter in common are significantly different. This is inferential shorthand,
    not evidence of equivalence.
    """
    nonsig = {frozenset((g, g)) for g in groups}
    for _, r in pairwise.iterrows():
        if pd.notna(r["p_holm"]) and r["p_holm"] >= alpha:
            nonsig.add(frozenset((r["group1"], r["group2"])))
    # All maximal cliques of the non-significance graph. With three groups this
    # is exact, transparent, and avoids a third-party CLD dependency.
    cliques = []
    for size in range(len(groups), 0, -1):
        for subset in itertools.combinations(groups, size):
            if all(frozenset(pair) in nonsig for pair in itertools.combinations(subset, 2)):
                s = set(subset)
                if not any(s < old for old in cliques):
                    cliques.append(s)
    # Keep only cliques needed to cover all non-significant pairs and singletons.
    targets = {frozenset((g,)) for g in groups}
    targets |= {p for p in nonsig if len(p) == 2}
    chosen, covered = [], set()
    while not targets.issubset(covered):
        best = max(cliques, key=lambda c: len(({frozenset((g,)) for g in c} |
                    {frozenset(p) for p in itertools.combinations(c, 2)}) - covered))
        chosen.append(best)
        covered |= {frozenset((g,)) for g in best}
        covered |= {frozenset(p) for p in itertools.combinations(best, 2)}
        cliques.remove(best)
    letters = {g: "" for g in groups}
    for i, clique in enumerate(chosen):
        letter = chr(ord("a") + i)
        for g in clique:
            letters[g] += letter
    return letters


def analyse_stratum(data: pd.DataFrame, outcome: str, label: str) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    d = data[["participant", "group", outcome]].dropna().copy()
    arrays = [d.loc[d.group == g, outcome].to_numpy() for g in GROUPS]
    if any(len(a) == 0 for a in arrays):
        return ({"stratum": label, "outcome": outcome, "H": np.nan, "df": 2,
                 "p_raw": np.nan, "epsilon_squared": np.nan, "n": len(d)},
                pd.DataFrame(), pd.DataFrame())
    if d[outcome].nunique() == 1:
        H, p = 0.0, 1.0
    else:
        H, p = stats.kruskal(*arrays)
    omnibus = {"stratum": label, "outcome": outcome, "H": H, "df": len(GROUPS)-1,
               "p_raw": p, "epsilon_squared": epsilon_squared_kw(H, len(d), len(GROUPS)),
               "n": len(d)}
    desc = (d.groupby("group", observed=True)[outcome]
              .agg(n="count", median="median", q1=lambda s: s.quantile(.25),
                   q3=lambda s: s.quantile(.75), mean="mean", sd="std")
              .reindex(GROUPS).reset_index())
    desc.insert(0, "outcome", outcome); desc.insert(0, "stratum", label)
    rows = []
    for g1, g2 in itertools.combinations(GROUPS, 2):
        x = d.loc[d.group == g1, outcome].to_numpy()
        y = d.loc[d.group == g2, outcome].to_numpy()
        test = stats.mannwhitneyu(x, y, alternative="two-sided", method="auto")
        rows.append({"stratum": label, "outcome": outcome, "group1": g1, "group2": g2,
                     "U": test.statistic, "p_raw": test.pvalue,
                     "rank_biserial_g1_vs_g2": rank_biserial(x, y),
                     "median_difference_g1_minus_g2": np.median(x)-np.median(y)})
    pair = pd.DataFrame(rows)
    pair["p_holm"] = holm(pair["p_raw"])
    pair["significant_holm"] = pair["p_holm"] < ALPHA
    letters = compact_letters(GROUPS, pair)
    desc["CLD"] = desc["group"].map(letters)
    return omnibus, pair, desc


def participant_summary(df: pd.DataFrame, outcome: str) -> pd.DataFrame:
    # Median is robust and ensures one independent value per participant.
    return (df.groupby(["participant", "group"], as_index=False, observed=True)[outcome]
              .median())


def aspect_mask(series: pd.Series, aspect: str) -> pd.Series:
    # Match enum token exactly, e.g. STRUCTURALTASKASPECT.COMP, avoiding iCOMP.
    return series.astype(str).str.contains(rf"STRUCTURALTASKASPECT\.{re.escape(aspect)}(?:\W|$)", regex=True)


def run_all(df: pd.DataFrame, output: str | Path) -> None:
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    omnibus_rows, pair_frames, desc_frames = [], [], []

    # Overall: participant-level median, not all 357 rows as independent cases.
    for outcome in ["TTU", "DOU"]:
        om, pw, ds = analyse_stratum(participant_summary(df, outcome), outcome, "Overall participant median")
        omnibus_rows.append(om); pair_frames.append(pw); desc_frames.append(ds)

    # Per task: naturally one row per participant.
    for task, sub in df.groupby("task", sort=True):
        for outcome in ["TTU", "DOU"]:
            om, pw, ds = analyse_stratum(sub, outcome, f"Task {task}")
            omnibus_rows.append(om); pair_frames.append(pw); desc_frames.append(ds)

    # Structural aspects: filter tasks, then aggregate within participant first.
    for aspect in ASPECTS:
        sub = df[aspect_mask(df["structural_aspects"], aspect)]
        if sub.empty:
            continue
        for outcome in ["TTU", "DOU"]:
            agg = participant_summary(sub, outcome)
            om, pw, ds = analyse_stratum(agg, outcome, f"Aspect {aspect}")
            om["tasks_included"] = ",".join(map(str, sorted(sub.task.unique())))
            omnibus_rows.append(om); pair_frames.append(pw); desc_frames.append(ds)

    omnibus = pd.DataFrame(omnibus_rows)
    pairwise = pd.concat([x for x in pair_frames if not x.empty], ignore_index=True)
    descriptives = pd.concat([x for x in desc_frames if not x.empty], ignore_index=True)

    # Holm adjustment across omnibus tests within each coherent family:
    # outcome x analysis type (overall/task/aspect). Pairwise Holm is within stratum.
    omnibus["family"] = np.select(
        [omnibus.stratum.str.startswith("Task"), omnibus.stratum.str.startswith("Aspect")],
        ["per_task", "per_aspect"], default="overall")
    omnibus["p_holm_family"] = np.nan
    for _, idx in omnibus.groupby(["outcome", "family"]).groups.items():
        omnibus.loc[idx, "p_holm_family"] = holm(omnibus.loc[idx, "p_raw"])
    omnibus["significant_raw"] = omnibus["p_raw"] < ALPHA
    omnibus["significant_holm_family"] = omnibus["p_holm_family"] < ALPHA

    omnibus.to_csv(out / "omnibus_tests.csv", index=False)
    pairwise.to_csv(out / "pairwise_tests.csv", index=False)
    descriptives.to_csv(out / "descriptives_and_cld.csv", index=False)

    audit = pd.DataFrame({
        "metric": ["rows", "participants", "tasks", "DOU_equal_1_count", "DOU_equal_1_percent"],
        "value": [len(df), df.participant.nunique(), df.task.nunique(),
                  int(df.DOU.eq(1).sum()), 100 * df.DOU.eq(1).mean()]})
    audit.to_csv(out / "data_audit.csv", index=False)

    # Concise console result.
    print("\nData audit")
    print(audit.to_string(index=False))
    print("\nOmnibus tests significant after family-wise Holm correction")
    sig = omnibus[omnibus.significant_holm_family]
    print(sig[["stratum", "outcome", "H", "p_raw", "p_holm_family", "epsilon_squared"]]
          .to_string(index=False) if len(sig) else "None")
    print(f"\nFull outputs written to: {out.resolve()}")




## Load and audit the data

The loader checks required columns, positive TTU, unique participant-group membership, and one row per participant-task.

In [2]:
DATA_PATH = 'study_data.csv'
df = load_data(DATA_PATH)
print(df.shape)
print(df.groupby('group')['participant'].nunique())
print('DOU = 1:', int(df.DOU.eq(1).sum()), f'({100*df.DOU.eq(1).mean():.1f}%)')
df.head()

(357, 12)
group
Dynamic sketch construction    7
Formula construction           7
Static sketch construction     7
Name: participant, dtype: int64
DOU = 1: 247 (69.2%)


,participant,id,group,TTU,DOU,structural_aspects,operation type,operand type,result type,Ref. direction,Ref. dispersion,task
0,p03,task_1,Dynamic sketch construction,28.0,1.0,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.CELL: 'Cell (Operand ty...,[<STRUCTURALTASKASPECT.VAL: 'Val (Result type)'>],[<STRUCTURALTASKASPECT.BACK: 'Back (Ref. direc...,[<STRUCTURALTASKASPECT.ON: 'On (Ref. dispersio...,1
1,p04,task_1,Dynamic sketch construction,26.0,1.0,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.CELL: 'Cell (Operand ty...,[<STRUCTURALTASKASPECT.VAL: 'Val (Result type)'>],[<STRUCTURALTASKASPECT.BACK: 'Back (Ref. direc...,[<STRUCTURALTASKASPECT.ON: 'On (Ref. dispersio...,1
2,p05,task_1,Dynamic sketch construction,32.0,1.0,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.CELL: 'Cell (Operand ty...,[<STRUCTURALTASKASPECT.VAL: 'Val (Result type)'>],[<STRUCTURALTASKASPECT.BACK: 'Back (Ref. direc...,[<STRUCTURALTASKASPECT.ON: 'On (Ref. dispersio...,1
3,p06,task_1,Dynamic sketch construction,64.0,1.0,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.CELL: 'Cell (Operand ty...,[<STRUCTURALTASKASPECT.VAL: 'Val (Result type)'>],[<STRUCTURALTASKASPECT.BACK: 'Back (Ref. direc...,[<STRUCTURALTASKASPECT.ON: 'On (Ref. dispersio...,1
4,p07,task_1,Dynamic sketch construction,24.0,1.0,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.COMP: 'Comp (Operation ...,[<STRUCTURALTASKASPECT.CELL: 'Cell (Operand ty...,[<STRUCTURALTASKASPECT.VAL: 'Val (Result type)'>],[<STRUCTURALTASKASPECT.BACK: 'Back (Ref. direc...,[<STRUCTURALTASKASPECT.ON: 'On (Ref. dispersio...,1


## Run the complete analysis

Outputs: `omnibus_tests.csv`, `pairwise_tests.csv`, `descriptives_and_cld.csv`, and `data_audit.csv`.

In [3]:
OUTPUT_DIR = 'study_results_notebook'
run_all(df, OUTPUT_DIR)


Data audit
             metric      value
               rows 357.000000
       participants  21.000000
              tasks  17.000000
  DOU_equal_1_count 247.000000
DOU_equal_1_percent  69.187675

Omnibus tests significant after family-wise Holm correction
stratum outcome         H    p_raw  p_holm_family  epsilon_squared
Task 15     DOU 13.604026 0.001112       0.018896         0.644668

Full outputs written to: /home/judith/Schreibtisch/angew. Informatik - system engineering/Study_DrawingAttention/study_results_notebook


## Inspect omnibus findings

Use the family-adjusted result as the primary multiplicity-controlled flag. With only seven participants per group, report exact p-values and effect sizes rather than significance alone.

In [4]:
omnibus = pd.read_csv(f'{OUTPUT_DIR}/omnibus_tests.csv')
omnibus.sort_values(['outcome','family','p_raw']).head(20)

,stratum,outcome,H,df,p_raw,epsilon_squared,n,tasks_included,family,p_holm_family,significant_raw,significant_holm_family
1,Overall participant median,DOU,1.057143,2,0.589446,0.000000e+00,21,NaN,overall,0.589446,False,False
41,Aspect LOOK,DOU,10.008511,2,0.006709,4.449173e-01,21,"14,15,16",per_aspect,0.100640,True,False
47,Aspect LIT,DOU,5.236648,2,0.072925,1.798138e-01,21,"3,8,10,15",per_aspect,1.000000,False,False
39,Aspect COND,DOU,2.353760,2,0.308239,1.965336e-02,21,"10,15,16,17",per_aspect,1.000000,False,False
65,Aspect CROSS,DOU,2.320802,2,0.313360,1.782233e-02,21,"12,13,14,17",per_aspect,1.000000,False,False
59,Aspect FORWARD,DOU,2.206655,2,0.331765,1.148083e-02,21,"4,12,13,14,15,16,17",per_aspect,1.000000,False,False
57,Aspect BACK,DOU,2.000000,2,0.367879,3.157968e-15,21,"1,2,3,5,6,7,8,9,10,11,12,13",per_aspect,1.000000,False,False
61,Aspect ON,DOU,2.000000,2,0.367879,3.157968e-15,21,"1,2,3,4,5,6,7,8,9,10,11,12,13,16",per_aspect,1.000000,False,False
63,Aspect OFF,DOU,2.000000,2,0.367879,3.157968e-15,21,"11,13,15",per_aspect,1.000000,False,False
55,Aspect LIST,DOU,1.562658,2,0.457797,0.000000e+00,21,"16,17",per_aspect,1.000000,False,False


## Inspect pairwise differences and compact letters

Pairwise tests are included for every stratum so effect sizes are always available. Interpret post-hoc significance mainly when the omnibus evidence is compelling.

In [5]:
pairwise = pd.read_csv(f'{OUTPUT_DIR}/pairwise_tests.csv')
descriptives = pd.read_csv(f'{OUTPUT_DIR}/descriptives_and_cld.csv')
pairwise.sort_values('p_holm').head(15)

,stratum,outcome,group1,group2,U,p_raw,rank_biserial_g1_vs_g2,median_difference_g1_minus_g2,p_holm,significant_holm
93,Task 15,DOU,Dynamic sketch construction,Formula construction,49.0,0.001027,1.000000,0.454545,0.003081,True
123,Aspect LOOK,DOU,Dynamic sketch construction,Formula construction,45.5,0.003676,0.857143,0.125000,0.011027,True
95,Task 15,DOU,Formula construction,Static sketch construction,6.0,0.017988,-0.755102,-0.454545,0.035975,True
19,Task 3,TTU,Dynamic sketch construction,Static sketch construction,43.0,0.020881,0.755102,10.000000,0.062642,False
52,Task 8,DOU,Dynamic sketch construction,Static sketch construction,41.5,0.023828,0.693878,0.142857,0.071483,False
84,Task 14,TTU,Dynamic sketch construction,Formula construction,7.0,0.029125,-0.714286,-49.000000,0.087375,False
100,Task 16,DOU,Dynamic sketch construction,Static sketch construction,38.5,0.030147,0.571429,0.125000,0.090441,False
141,Aspect LIT,DOU,Dynamic sketch construction,Formula construction,40.0,0.033792,0.632653,0.162500,0.101376,False
53,Task 8,DOU,Formula construction,Static sketch construction,39.0,0.053643,0.591837,0.142857,0.107285,False
97,Task 16,TTU,Dynamic sketch construction,Static sketch construction,41.0,0.037879,0.673469,33.000000,0.113636,False


In [6]:
descriptives[descriptives['stratum'].eq('Task 15') & descriptives['outcome'].eq('DOU')]

,stratum,outcome,group,n,median,q1,q3,mean,sd,CLD
93,Task 15,DOU,Dynamic sketch construction,7,1.000000,1.000000,1.000000,1.000000,0.000000,a
94,Task 15,DOU,Formula construction,7,0.545455,0.318182,0.772727,0.519481,0.358733,b
95,Task 15,DOU,Static sketch construction,7,1.000000,0.909091,1.000000,0.909091,0.174078,a


## Reporting template

For each analysis, report group medians and IQRs, then: “A Kruskal-Wallis test [did/did not] indicate a group difference, H(2)=..., p=..., Holm-adjusted p=..., epsilon-squared=....” Where warranted, add Holm-adjusted pairwise Mann-Whitney tests with signed rank-biserial correlations and the compact-letter display.

## Important limitation

Participant-level aggregation asks whether groups differ in typical overall or aspect-specific performance. It does not estimate a group-by-task interaction. A formal interaction needs a mixed-effects or rank-based repeated-measures model, preferably with more participants. Per-task analyses address where differences occur but have low power and require multiplicity control.